# Барановський Богдан ФБ-44 Лабораторна робота №2 - Наука про дані: підготовчий етап

## **Частина 1.** Завдання:
* Створити віртуальне середовище (venv) в якому будуть встановлені всі необхідні бібліотеки та налаштування для даної лабораторної роботи;
* Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних;
  * Приклад посилання для області №8: https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID=8&year1=1981&year2=2024&type=Mean
  * Якщо вказати provinceID=0, сформується вибірка в середньому по Україні. Її завантажувати не потрібно;
* Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області (див. How to Handle Missing Data in Python?)
* Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 
* Реалізувати процедури для формування вибірок (https://www.geeksforgeeks.org/pandas/ways-to-filter-pandas-dataframe-by-column-values/)  наступного виду:
    * Ряд VHI для області за вказаний рік;
    * Ряд VHI за вказаний діапазон років для вказаних областей;
    * Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани;
    * Інші вибірки на вимогу викладача.


**Перейдемо до виконання завдання** <br> 
Створено віртуальне середовище venv та всі потрібні бібліотеки додані в requirements.txt <br>
Для завантаження тестових структурованих файлів, інсталюємо наступні бібліотеки:

In [1]:
import pandas as pd
import urllib.request
import os
from datetime import datetime

Тепер створюємо процедуру завантаження файлів з NOAA для кожної області:

In [2]:
if not os.path.exists("vhi_data"):
    os.makedirs("vhi_data")

def download_vhi(province_id):
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    
    for f in os.listdir("vhi_data"):
        if f.startswith(f"vhi_id_{province_id}_"):
            print(f"{province_id} already downloaded.")
            return f"vhi_data/{f}"
    
    now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"vhi_data/vhi_id_{province_id}_{now}.csv"
    
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    response = urllib.request.urlopen(req)
    data = response.read().decode('utf-8')
    
    with open(filename, 'w') as f:
        f.write(data)
        
    print(f"File saved: {filename}")
    return filename

files = []
for i in range(1, 28):
    path = download_vhi(i)
    files.append((i, path))

1 already downloaded.
2 already downloaded.
3 already downloaded.
4 already downloaded.
5 already downloaded.
6 already downloaded.
7 already downloaded.
8 already downloaded.
9 already downloaded.
10 already downloaded.
11 already downloaded.
12 already downloaded.
13 already downloaded.
14 already downloaded.
15 already downloaded.
16 already downloaded.
17 already downloaded.
18 already downloaded.
19 already downloaded.
20 already downloaded.
21 already downloaded.
22 already downloaded.
23 already downloaded.
24 already downloaded.
25 already downloaded.
26 already downloaded.
27 already downloaded.


Як бачимо, все успішно завантажено: <br> <img src="screenshots/img1.png" width="25%">

Далі зчитуємо завантажені текстові файли у pandas dataframe та здійснюємо data cleaning.

In [3]:
def clean_data(files_list):
    cols = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
    df_list = []

    for pid, path in files_list:
        df = pd.read_csv(path, header=1, names=cols, skip_blank_lines=True)
        
        if 'empty' in df.columns:
            df = df.drop(columns=['empty'])
        
        df = df.dropna(subset=['VHI'])
        df = df[df['VHI'] != -1]
        
        df['Year'] = df['Year'].astype(str).str.replace(r'<[^>]*>', '', regex=True)
        df = df[df['Year'].str.strip() != '']
        df['Year'] = df['Year'].astype(int)
        
        df['Area_ID'] = pid
        df_list.append(df)

    return pd.concat(df_list, ignore_index=True)

df = clean_data(files)
display(df.head())

,Year,Week,SMN,SMT,VCI,TCI,VHI,Area_ID
0,1982,1.0,0.053,260.31,45.01,39.46,42.23,1
1,1982,2.0,0.054,262.29,46.83,31.75,39.29,1
2,1982,3.0,0.055,263.82,48.13,27.24,37.68,1
3,1982,4.0,0.053,265.33,46.09,23.91,35.00,1
4,1982,5.0,0.050,265.66,41.46,26.65,34.06,1


Додаємо стовпчики з назвою та індексом області і реалізуємо процедуру зміни індексів так, щоб області індексувалися за українською абеткою.

In [4]:
index_dict = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
    11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
    20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
}

names_dict = {
    1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька',
    5: 'Житомирська', 6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська',
    9: 'Київська', 10: 'Кіровоградська', 11: 'Луганська', 12: 'Львівська',
    13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська', 16: 'Рівненська',
    17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
    21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська',
    25: 'Республіка Крим', 26: 'м. Київ', 27: 'м. Севастополь'
}

def update_indexes(data):
    data['Area_ID'] = data['Area_ID'].map(index_dict)
    data['Area_Name'] = data['Area_ID'].map(names_dict)
    return data[['Year', 'Week', 'Area_ID', 'Area_Name', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']]

df = update_indexes(df)
print("Data after reindexing:")
display(df.head())

Data after reindexing:


,Year,Week,Area_ID,Area_Name,SMN,SMT,VCI,TCI,VHI
0,1982,1.0,22,Черкаська,0.053,260.31,45.01,39.46,42.23
1,1982,2.0,22,Черкаська,0.054,262.29,46.83,31.75,39.29
2,1982,3.0,22,Черкаська,0.055,263.82,48.13,27.24,37.68
3,1982,4.0,22,Черкаська,0.053,265.33,46.09,23.91,35.00
4,1982,5.0,22,Черкаська,0.050,265.66,41.46,26.65,34.06


Реалізовуэмо процедури для формування вибірок наступного виду:
* Ряд VHI для області за вказаний рік;
* Ряд VHI за вказаний діапазон років для вказаних областей;
* Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани;
* Інші вибірки на вимогу викладача.

In [5]:
def get_vhi_by_year(data, area, year):
    return data[(data['Area_ID'] == area) & (data['Year'] == year)][['Week', 'VHI']]

print("VHI for Area 1 in 2024:")
display(get_vhi_by_year(df, 1, 2024).head())


def get_vhi_range(data, areas, year1, year2):
    return data[(data['Area_ID'].isin(areas)) & (data['Year'] >= year1) & (data['Year'] <= year2)][['Year', 'Week', 'Area_Name', 'VHI']]

print("\nVHI for Areas 1 and 2 from 2022 to 2024:")
display(get_vhi_range(df, [1, 2], 2022, 2024).head())


def get_extremes(data, areas, year1, year2):
    filtered = data[(data['Area_ID'].isin(areas)) & (data['Year'] >= year1) & (data['Year'] <= year2)]
    
    return {
        'Min': filtered['VHI'].min(),
        'Max': filtered['VHI'].max(),
        'Mean': round(filtered['VHI'].mean(), 2),
        'Median': filtered['VHI'].median()
    }

print("\nExtremes for Areas 1 and 2 (2022-2024):")
print(get_extremes(df, [1, 2], 2022, 2024))

VHI for Area 1 in 2024:


,Week,VHI
52412,1.0,42.15
52413,2.0,44.12
52414,3.0,45.80
52415,4.0,46.04
52416,5.0,45.22



VHI for Areas 1 and 2 from 2022 to 2024:


,Year,Week,Area_Name,VHI
52308,2022,1.0,Вінницька,49.12
52309,2022,2.0,Вінницька,50.20
52310,2022,3.0,Вінницька,51.26
52311,2022,4.0,Вінницька,50.79
52312,2022,5.0,Вінницька,47.71



Extremes for Areas 1 and 2 (2022-2024):
{'Min': np.float64(28.83), 'Max': np.float64(62.58), 'Mean': np.float64(47.99), 'Median': np.float64(49.105000000000004)}
